In [4]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
import io
import sys
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import seaborn as sns
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from sklearn.feature_selection import SelectKBest, f_classif

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from IPython.display import display

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score

from typing import Tuple
from sklearn.base import RegressorMixin
from typing import Tuple, List
from sklearn.pipeline import Pipeline
from sklearn.base import RegressorMixin
from typing import Optional
from sklearn.base import ClassifierMixin

from xgboost import XGBClassifier
from sklearn.ensemble import StackingClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

from typing import Tuple, List, Dict
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_curve, auc
from sklearn.metrics import precision_recall_curve, average_precision_score

from sklearn.pipeline import make_pipeline
import optuna.visualization as vis
from statsmodels.tsa.arima.model import ARIMA


from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX

from ts_automator import run_timeseries



In [2]:
"""
Utility module to automate common univariate time‑series workflows inspired by patron.py.

Main entry point
----------------
run_timeseries(
    input_data: Union[str, pd.DataFrame],  # path to csv or a DataFrame already in memory
    date_col: str,                         # name of the column with the timestamps
    target_col: str,                       # series to analyze / forecast
    method: str = "plot",                # one of ['plot', 'decompose', 'auto_arima', 'skforecast_rf']
    freq: Optional[str] = None,            # pandas offset alias (e.g. 'D', 'MS')
    test_size: float = 0.2,                # fraction of observations for test when relevant
    model_params: Optional[dict] = None,   # kwargs forwarded to the model constructor
    **kwargs                               # extra control depending on the method
) -> Any

Supported methods
~~~~~~~~~~~~~~~~~
* **plot** – Simple line plot of the series.
* **decompose** – Seasonal–Trend decomposition using `statsmodels.seasonal_decompose`.
* **auto_arima** – Automatic ARIMA order search & forecast via *pmdarima*.
* **skforecast_rf** – Recursive multi‑step forecast with Random‑Forest using *skforecast*.

The function handles loading, resampling, fitting, plotting, and returns the
most relevant result for the chosen workflow (e.g., decomposition object,
forecast series, or (preds, mse)).

Example
~~~~~~~
```python
from ts_automator import run_timeseries

# Seasonal decomposition
run_timeseries('mydata.csv', date_col='fecha', target_col='y',
              method='decompose', freq='MS', seasonal_period=12)

# ARIMA forecast 24 months ahead
run_timeseries('mydata.csv', 'fecha', 'y',
              method='auto_arima', freq='MS', n_periods=24,
              model_params={'m': 12, 'seasonal': True})
```"""

from __future__ import annotations

from typing import Any, Optional, Union, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.seasonal import seasonal_decompose

# Optional, only loaded when necessary to avoid heavy imports if a user only wants plotting
try:
    import pmdarima as pm  # type: ignore
except ModuleNotFoundError:  # pragma: no cover – tolerates missing optional deps
    pm = None

try:
    from skforecast.ForecasterRecursive import ForecasterRecursive  # type: ignore
except ModuleNotFoundError:  # pragma: no cover
    ForecasterRecursive = None  # type: ignore


def _load_series(
    data: Union[str, pd.DataFrame],
    date_col: str,
    target_col: str,
    freq: Optional[str] = None,
) -> pd.Series:
    """Parse *data* (csv path or in‑memory DataFrame) into a time‑indexed Series."""
    if isinstance(data, pd.DataFrame):
        df = data.copy()
    else:
        df = pd.read_csv(data)

    if date_col not in df.columns:
        raise ValueError(f"Column '{date_col}' not found in the provided data.")
    if target_col not in df.columns:
        raise ValueError(f"Column '{target_col}' not found in the provided data.")

    df[date_col] = pd.to_datetime(df[date_col])
    ts = df.set_index(date_col)[target_col].sort_index()
    if freq is not None:
        ts = ts.asfreq(freq)
    return ts


def run_timeseries(
    input_data: Union[str, pd.DataFrame],
    date_col: str,
    target_col: str,
    *,
    method: str = "plot",
    freq: Optional[str] = None,
    test_size: float = 0.2,
    model_params: Optional[dict[str, Any]] = None,
    **kwargs: Any,
) -> Any:
    """Automate different time‑series pipelines with one function call.

    Parameters
    ----------
    input_data : Union[str, pd.DataFrame]
        Path to *.csv* file **or** a ready *DataFrame*.
    date_col, target_col : str
        Column names for the timestamp and the target series.
    method : str, default "plot"
        Workflow to run. One of *plot*, *decompose*, *auto_arima*, *skforecast_rf*.
    freq : str, optional
        If provided, resamples / coerces the index to this frequency using *asfreq*.
    test_size : float, default 0.2
        Fraction of observations reserved for testing when the chosen method needs a split.
    model_params : dict, optional
        Hyper‑parameters forwarded to the underlying model constructor (ARIMA or RF).
    **kwargs : Any
        Extra knobs that depend on the selected *method*.

    Returns
    -------
    Depends on the *method*:
        * plot – the Series itself (after any resampling)
        * decompose – *statsmodels* DecomposeResult
        * auto_arima – forecast *pd.Series*
        * skforecast_rf – Tuple[pd.Series, float] → (predictions, test_MSE)
    """

    method = method.lower()
    y = _load_series(input_data, date_col, target_col, freq=freq)

    if method == "plot":
        ax = y.plot(figsize=(10, 4), title=f"{target_col} over time", lw=1)
        ax.set_xlabel("")
        plt.show()
        return y

    if method == "decompose":
        seasonal_period = kwargs.get("seasonal_period")
        result = seasonal_decompose(
            y,
            model=kwargs.get("model", "additive"),
            period=seasonal_period,
            extrapolate_trend="freq",
        )
        result.plot()
        plt.show()
        return result

    if method == "auto_arima":
        if pm is None:
            raise ImportError("pmdarima is not installed. Install it to use auto_arima method.")
        model = pm.auto_arima(y, **(model_params or {}))
        print(model.summary())

        n_periods = int(kwargs.get("n_periods", 12))
        fc, confint = model.predict(n_periods=n_periods, return_conf_int=True)
        idx = pd.date_range(y.index[-1], periods=n_periods + 1, freq=y.index.freq)[1:]
        fc_series = pd.Series(fc, index=idx, name="forecast")
        lower, upper = confint.T

        plt.figure(figsize=(10, 4))
        plt.plot(y, label="historical")
        plt.plot(fc_series, label="forecast")
        plt.fill_between(idx, lower, upper, alpha=0.2)
        plt.legend()
        plt.show()
        return fc_series

    if method == "skforecast_rf":
        if ForecasterRecursive is None:
            raise ImportError("skforecast is not installed. Install it to use skforecast_rf method.")
        n_test = int(len(y) * test_size)
        y_train, y_test = y.iloc[:-n_test], y.iloc[-n_test:]
        steps = len(y_test)
        rf_params = model_params or {}
        regressor = RandomForestRegressor(random_state=123, **rf_params)
        forecaster = ForecasterRecursive(regressor=regressor, lags=kwargs.get("lags", 10))
        forecaster.fit(y=y_train)
        preds = forecaster.predict(steps=steps)
        mse = mean_squared_error(y_test, preds)
        print(f"Test MSE: {mse:.4f}")

        plt.figure(figsize=(10, 4))
        y_train.plot(label="train")
        y_test.plot(label="test")
        preds.plot(label="forecast")
        plt.legend()
        plt.show()
        return preds, mse

    raise ValueError(
        "method must be one of 'plot', 'decompose', 'auto_arima', 'skforecast_rf'"
    )


if __name__ == "__main__":  # pragma: no cover – quick smoke test
    import argparse

    parser = argparse.ArgumentParser(description="Quick CLI wrapper for run_timeseries().")
    parser.add_argument("data", help="CSV file to load")
    parser.add_argument("date_col")
    parser.add_argument("target_col")
    parser.add_argument("--method", default="plot")
    parser.add_argument("--freq", default=None)
    parser.add_argument("--n_periods", type=int, default=12)
    args = parser.parse_args()

    run_timeseries(
        args.data,
        args.date_col,
        args.target_col,
        method=args.method,
        freq=args.freq,
        n_periods=args.n_periods,
    )


usage: ipykernel_launcher.py [-h] [--method METHOD] [--freq FREQ]
                             [--n_periods N_PERIODS]
                             data date_col target_col
ipykernel_launcher.py: error: the following arguments are required: date_col, target_col


SystemExit: 2

In [3]:
# 1. Gráfico y descomposición mensual
run_timeseries('Registros_Condiciones.csv',
               date_col='fecha',
               target_col='temperatura',
               method='decompose',
               freq='MS',
               seasonal_period=12)

# 2. Pronóstico ARIMA con 24 pasos
run_timeseries(df,
               date_col='timestamp',
               target_col='y',
               method='auto_arima',
               n_periods=24)


ModuleNotFoundError: No module named 'ts_automator'